In [5]:
import xarray as xr
import numpy as np
import rasterio
import geopandas as gpd
import pandas as pd
import boto3
import os 
import time
import rioxarray as rxr

from rasterio.warp import reproject, Resampling
from scipy.ndimage import distance_transform_edt
from skimage.measure import block_reduce
import rioxarray as rxr
os.environ['AWS_PROFILE'] = 'ctrees'
os.environ['GDAL_DISABLE_READDIR_ON_OPEN'] = 'YES'
os.environ['VSI_CURL_ALLOW_CREDENTIALS'] = 'YES'

from tessera_utils import get_tile_bbox, get_tessera_center_from_bbox

In [6]:
def tessera_prediction_paths(planet_s3_path, year):
    """
    Generate the S3 path for the predicted tile based on Planet tile ID.
    """
    pred_bucket = 'ctrees-development'
    pred_prefix = 'activity/tessera/logging_pantropical_v1/predictions/pantropical_v1/'
    planet_tile_bbox = get_tile_bbox(planet_s3_path)
    tessera_ids = get_tessera_center_from_bbox(*planet_tile_bbox)

    tessera_keys_dict = {}
    for t in tessera_ids:
        t_string = f'{t[0]}_{t[1]}'
        key = f'tile_{t_string}/tile_{t_string}_{year}_p.tif'
        tessera_keys_dict[t_string] = f's3://{pred_bucket}/{pred_prefix}{key}'

    return tessera_keys_dict
    # tessera_keys = [f'tile_{t[0]}_{t[1]}/tile_{t[0]}_{t[1]}_{year}_p.tif' for t in tessera_ids]
    # return [f's3://{pred_bucket}/{pred_prefix}{name}' for name in tessera_keys]


In [7]:
testing_tile_path = 's3://ctrees-development/activity/tessera/planet_nicfi_logging_tessera_pantropical_v1_testing.gpkg'
testing_tiles = gpd.read_file(testing_tile_path)
tiles_id = testing_tiles['name'].tolist()
print(f"There are {len(tiles_id)} testing tiles.")
tiles_id

There are 10 testing tiles.


['0705-1003',
 '0703-0985',
 '0649-0971',
 '0683-0968',
 '0694-0962',
 '0714-0956',
 '1107-1042',
 '1094-1021',
 '1672-1038',
 '1688-1036']

In [8]:
paths_dict = {}
product = 'logging'
planet_prefix = f's3://ctrees-activity-product/{product}/v1_2024_update/cumulative/2015-12-01-2025-01-01/'
years = [2024] #range(2017, 2025+1)
predictions_prefix = f's3://ctrees-development/activity/tessera/logging_pantropical_v1/predictions/pantropical_v1/'
for tile_id in tiles_id:
    
    filename_planet = f'{product}_cumulative_PAGE_{tile_id}_START_2015-12-01_END_2025-01-01.tif'
    planet_s3_path = planet_prefix + filename_planet
    predictions_s3_paths = [tessera_prediction_paths(planet_s3_path, year) for year in years]
    
    paths_dict[tile_id] = {
        'planet_s3_path': planet_s3_path,
        'predictions_s3_paths': predictions_s3_paths
    }

paths_dict

{'0705-1003': {'planet_s3_path': 's3://ctrees-activity-product/logging/v1_2024_update/cumulative/2015-12-01-2025-01-01/logging_cumulative_PAGE_0705-1003_START_2015-12-01_END_2025-01-01.tif',
  'predictions_s3_paths': [{'-56.05_-3.65': 's3://ctrees-development/activity/tessera/logging_pantropical_v1/predictions/pantropical_v1/tile_-56.05_-3.65/tile_-56.05_-3.65_2024_p.tif',
    '-56.05_-3.55': 's3://ctrees-development/activity/tessera/logging_pantropical_v1/predictions/pantropical_v1/tile_-56.05_-3.55/tile_-56.05_-3.55_2024_p.tif',
    '-55.95_-3.65': 's3://ctrees-development/activity/tessera/logging_pantropical_v1/predictions/pantropical_v1/tile_-55.95_-3.65/tile_-55.95_-3.65_2024_p.tif',
    '-55.95_-3.55': 's3://ctrees-development/activity/tessera/logging_pantropical_v1/predictions/pantropical_v1/tile_-55.95_-3.55/tile_-55.95_-3.55_2024_p.tif',
    '-55.85_-3.65': 's3://ctrees-development/activity/tessera/logging_pantropical_v1/predictions/pantropical_v1/tile_-55.85_-3.65/tile_-55.85

In [9]:
# export a json file with the paths_dict
import json
with open('paths_dict.json', 'w') as f:    json.dump(paths_dict, f) 

In [5]:
def load_label_by_tessera_bbox(label_da, tessera_bbox):
    """Load the label data array clipped to the tessera bounding box."""
    # reproject label_da to EPSG:4326 if needed
    if label_da.rio.crs is not None and label_da.rio.crs.to_epsg() != 4326:
        label_da = label_da.rio.reproject("EPSG:4326", resampling=Resampling.nearest)   
    # read label_da by tessera_bbox
    label_da_clipped = label_da.rio.clip_box(*tessera_bbox)
    return label_da_clipped


In [6]:
TESSERA_THRESHOLD = 0.8
def select_samples(final_labels, prediction_da, tile_id, tessera_id, num_samples=5000):
    l_mask = (final_labels == 1) & (prediction_da >= TESSERA_THRESHOLD)
    nl_mask = (final_labels == 0) & (prediction_da < TESSERA_THRESHOLD)

    l_indices = np.argwhere(l_mask.values)
    nl_indices = np.argwhere(nl_mask.values)

    if len(l_indices) > num_samples:
        sampled_l_indices = l_indices[np.random.choice(l_indices.shape[0], num_samples, replace=False)]
    else:
        sampled_l_indices = l_indices

    if len(nl_indices) > num_samples:
        sampled_nl_indices = nl_indices[np.random.choice(nl_indices.shape[0], num_samples, replace=False)]
    else:
        sampled_nl_indices = nl_indices

    sampled_data = []
    for row, col in sampled_l_indices:
        sampled_data.append({
            'row': row,
            'col': col,
            'lon': float(prediction_da.x.values[col]),
            'lat': float(prediction_da.y.values[row]),
            'class': int(final_labels.values[row, col]),
            'prediction_value': float(prediction_da.values[row, col]),
            'planet_id': tile_id,
            'tessera_id': tessera_id,
            'tessera_crs': prediction_da.rio.crs.to_string() if prediction_da.rio.crs is not None else 'unknown'
        })

    for row, col in sampled_nl_indices:
        sampled_data.append({
            'row': row,
            'col': col,
            'lon': float(prediction_da.x.values[col]),
            'lat': float(prediction_da.y.values[row]),
            'class': int(final_labels.values[row, col]),
            'prediction_value': float(prediction_da.values[row, col]),
            'planet_id': tile_id,
            'tessera_id': tessera_id,
            'tessera_crs': prediction_da.rio.crs.to_string() if prediction_da.rio.crs is not None else 'unknown'
        })

    samples_df = pd.DataFrame(sampled_data)
    return samples_df

def export_samples_to_s3(samples_df, tile_id):
    output_bucket = 'ctrees-development'
    file_name = f'{tile_id}_samples.parquet'
    output_prefix = f'activity/tessera/logging_pantropical_v1/reinforced_samples'
    output_path = f's3://{output_bucket}/{output_prefix}/{file_name}'

    samples_df.to_parquet(
        output_path,
        storage_options={'profile': 'ctrees'}
    )
    print(f"Exported sampled data to: {output_path}")


def process_labels(label_da,prediction_path):
    """
    Process the labels by reprojecting, creating masks,
      applying a buffer, and preparing the final labels for output.
    Args:
        label_da (xarray.DataArray): The label DataArray.
        prediction_path (str): The file path to the prediction raster.
    Returns:
        xarray.DataArray: The processed label DataArray ready for output.
    """

    tessera_bbox = get_tile_bbox(prediction_path)
    label_da = load_label_by_tessera_bbox(label_da, tessera_bbox)
    prediction_da = rxr.open_rasterio(prediction_path, masked=True).squeeze()

    # reproject and match predicton projections
    labels = label_da.rio.reproject_match(prediction_da, resampling=Resampling.nearest) 

    #logic
    l = (labels == 240) | (labels == 245)
    nl = (labels == 0)  

    # masks
    no_logging_pixels = labels.where(nl, 255) # keeps 0 no logging, else no data (255)
    no_valid_pixels = labels.where(l | nl, 255) # keeps pixels that are logging and not logging, else no data (255)

    # create buffer to reduces loggin edges
    no_logging = no_logging_pixels.where(no_logging_pixels == 255, 1) # 1-> no loggin, 0 --> everything else
    no_logging_binary = no_logging.where(no_logging == 1, 0)

    # Apply buffer:
    N = 3
    buff = (
        no_logging_binary
        .rolling(x=2*N+1, center=True, min_periods=1).max()
        .rolling(y=2*N+1, center=True, min_periods=1).max()
    ) == 1

    # invert values so 1--> loggin and others, 0 --> no logging
    invr_buff = ~buff
    invr_buff = invr_buff.astype('uint8') 

    # set other values no data value (255)
    final_labels = invr_buff.where(no_valid_pixels !=255, 255)

    return final_labels, prediction_da

def get_samples(tile_id, paths_dict):
    """Get samples for a given Planet tile ID using the provided paths dictionary.
    Export by Tesserra tile that overlap Planet tile. So later is eassy to extract the embeddigns.
    """
    inputs = paths_dict[tile_id]
    planet_file_path = inputs['planet_s3_path']
    prediction_paths_dict = inputs['predictions_s3_paths'][0]

    label_da = rxr.open_rasterio(planet_file_path, masked=True).squeeze()

    for tessera_id, pred_path in prediction_paths_dict.items():
        final_labels, prediction_da = process_labels(label_da, pred_path)
        samples_df = select_samples(final_labels, prediction_da, tile_id, tessera_id)
        # save the samples per tessera tile, not planet. So later is eassy to extract the embeddigns.
        export_samples_to_s3(samples_df, tessera_id)

        

In [7]:
for tile_id in tiles_id:
    t0 = time.time()
    print(f"Processing tile: {tile_id}")
    get_samples(tile_id, paths_dict)
    t1 = time.time()
    print(f"Finished processing tile: {tile_id} in {t1 - t0:.2f} seconds")

Processing tile: 0705-1003
Exported sampled data to: s3://ctrees-development/activity/tessera/logging_pantropical_v1/reinforced_samples/-56.05_-3.65_samples.parquet
Exported sampled data to: s3://ctrees-development/activity/tessera/logging_pantropical_v1/reinforced_samples/-56.05_-3.55_samples.parquet
Exported sampled data to: s3://ctrees-development/activity/tessera/logging_pantropical_v1/reinforced_samples/-55.95_-3.65_samples.parquet
Exported sampled data to: s3://ctrees-development/activity/tessera/logging_pantropical_v1/reinforced_samples/-55.95_-3.55_samples.parquet
Exported sampled data to: s3://ctrees-development/activity/tessera/logging_pantropical_v1/reinforced_samples/-55.85_-3.65_samples.parquet
Exported sampled data to: s3://ctrees-development/activity/tessera/logging_pantropical_v1/reinforced_samples/-55.85_-3.55_samples.parquet
Finished processing tile: 0705-1003 in 28.29 seconds
Processing tile: 0703-0985
Exported sampled data to: s3://ctrees-development/activity/tesser

In [39]:
test_tile ='0705-1003'
get_samples(test_tile, paths_dict)

Exported sampled data to: s3://ctrees-development/activity/tessera/logging_pantropical_v1/reinforced_samples/-56.05_-3.65_samples.parquet
Exported sampled data to: s3://ctrees-development/activity/tessera/logging_pantropical_v1/reinforced_samples/-56.05_-3.55_samples.parquet
Exported sampled data to: s3://ctrees-development/activity/tessera/logging_pantropical_v1/reinforced_samples/-55.95_-3.65_samples.parquet
Exported sampled data to: s3://ctrees-development/activity/tessera/logging_pantropical_v1/reinforced_samples/-55.95_-3.55_samples.parquet
Exported sampled data to: s3://ctrees-development/activity/tessera/logging_pantropical_v1/reinforced_samples/-55.85_-3.65_samples.parquet
Exported sampled data to: s3://ctrees-development/activity/tessera/logging_pantropical_v1/reinforced_samples/-55.85_-3.55_samples.parquet


In [40]:
parquet_path = 's3://ctrees-development/activity/tessera/logging_pantropical_v1/reinforced_samples/-56.05_-3.65_samples.parquet'
# open parquet
samples_df = pd.read_parquet(parquet_path, storage_options={'profile': 'ctrees'})
print(samples_df.head())

   row  col            lon           lat  class  prediction_value  planet_id  \
0  774  929  609239.217594  9.594292e+06      1          0.851228  0705-1003   
1  798  854  608489.217594  9.594052e+06      1          0.956132  0705-1003   
2  806  856  608509.217594  9.593972e+06      1          0.932168  0705-1003   
3  952  735  607299.217594  9.592512e+06      1          0.958982  0705-1003   
4  107  374  603689.217594  9.600962e+06      1          0.919382  0705-1003   

  tessera_crs  
0  EPSG:32721  
1  EPSG:32721  
2  EPSG:32721  
3  EPSG:32721  
4  EPSG:32721  


In [43]:
parquet_path = 's3://ctrees-development/activity/tessera/logging_pantropical_v1/extracted_embeddings/embeddings_PAGE_0602-0978.parquet'
# open parquet
samples_df = pd.read_parquet(parquet_path, storage_options={'profile': 'ctrees'})
print(samples_df.head())

   class   row   col        x_proj         y_proj        lon       lat  \
0      1   765   153 -8.256178e+06 -887868.633860 -74.166512 -7.950224   
1      1   739  1029 -8.247809e+06 -887620.213518 -74.091325 -7.948014   
2      1   617  1117 -8.246968e+06 -886454.548837 -74.083772 -7.937643   
3      1  1851   717 -8.250790e+06 -898244.960448 -74.118104 -8.042530   
4      1   689    87 -8.256809e+06 -887142.482091 -74.172177 -7.943764   

  planet_crs                                      source_raster  \
0  EPSG:3857  s3://ctrees-activity-product/logging/v1_2024_u...   
1  EPSG:3857  s3://ctrees-activity-product/logging/v1_2024_u...   
2  EPSG:3857  s3://ctrees-activity-product/logging/v1_2024_u...   
3  EPSG:3857  s3://ctrees-activity-product/logging/v1_2024_u...   
4  EPSG:3857  s3://ctrees-activity-product/logging/v1_2024_u...   

   tessera_tile_lon  ...   emb_123   emb_124   emb_125   emb_126   emb_127  \
0            -74.15  ...  0.142917 -2.858346 -0.357293  3.001263 -0.571669